In [32]:
import timeit
import os
import xarray as xr
from smmregrid import cdo_generate_weights, Regridder
from cdo import Cdo
import pandas as pd
import copy
cdo = Cdo()
import dask
dask.config.set(scheduler="synchronous")

# where and which the data are
indir='./data'
# filelist = ['nemo-eORCA12_hpz10_nested_oce.nc', 'tas-healpix2.nc', 'onlytos-ipsl.nc','tas-ecearth.nc', 
#             '2t-era5.nc','tos-fesom.nc', 'ua-ecearth.nc', 'mix-cesm.nc']#,'era5-mon.nc'] # the last is not available on github
filelist = ['nemo-eORCA12_hpz10_nested_oce.nc']
tfile = os.path.join(indir, 'r360x180.nc')

# method for remapping
methods = ['nn','con','bil']
accesses = ['Dataset', 'DataArray']


# create an iterable dictionary, and clean cases where we know CDO does not work
defdict = {'methods': methods, 'accesses': accesses, 'extra': '', 'chunks': None, 'options': '', 'var': None}
base = {k: copy.deepcopy(defdict) for k in filelist}
if 'tos-fesom.nc' in filelist:
    base['tos-fesom.nc']['methods'].remove('bil')
if 'tas-healpix2.nc' in filelist:
    base['tas-healpix2.nc']['methods'].remove('bil')
if 'lsm-ifs.grb' in filelist:
    base['lsm-ifs.grb']['extra'] = '-setgridtype,regular'
    base['lsm-ifs.grb']['methods'].remove('bil')
    base['lsm-ifs.grb']['methods'].remove('con')
if 'mix-cesm.nc' in filelist:
    base['mix-cesm.nc']['accesses'].remove('DataArray')
if 'era5-mon.nc' in filelist:
    base['era5-mon.nc']['chunks'] = {'time': 12}
if 'ua-ecearth.nc' in filelist:
    base['ua-ecearth.nc']['chunks'] = {'plev': 3}
if 'nemo-eORCA12_hpz10_nested_oce.nc' in filelist:
    base['nemo-eORCA12_hpz10_nested_oce.nc']['options'] = '--force'
    # We set the var to be used for the regridding
    base['nemo-eORCA12_hpz10_nested_oce.nc']['var'] = ['avg_sos']



In [35]:
data =[]
for filein in base.keys():
    print(f"Analyzing {filein} with options {base[filein]}")
    nr = 20

    # CDO
    wfile = cdo.gencon(tfile, input = os.path.join(indir,filein), options = base[filein].get('options', ''))
    ccdo = timeit.timeit(lambda: cdo.remap(tfile + ',' + wfile, input = os.path.join(indir,filein), returnXDataset = True).load(), number = nr)
    cdonoload = timeit.timeit(lambda: cdo.remap(tfile + ',' + wfile, input = os.path.join(indir,filein), returnXDataset = True), number = nr)

    # SMM: load field and weights, initialize regridder
    xfield = xr.open_mfdataset(os.path.join(indir,filein)).load()
    wfield = cdo_generate_weights(os.path.join(indir,filein), tfile, method = 'con', cdo_options=base[filein].get('options', '')).load()
    interpolator = Regridder(weights=wfield)
 
    # var as the one which have time and not have bnds, pick the first one
    myvar = [var for var in xfield.data_vars 
             if 'time' in xfield[var].dims and 'bnds' not in xfield[var].dims]
    if len(myvar) == 0:
        myvar = base[filein].get('var', None)
    print(f"Using variable {myvar} for regridding")
   
    # dataset infos
    nrecords = xfield[myvar[0]].shape
    nvars = len(myvar)


    sset =      timeit.timeit(lambda: interpolator.regrid(xfield).load(), number = nr)
    arr =       timeit.timeit(lambda: interpolator.regrid(xfield[myvar[0]]).load(), number = nr)
    arrnoload = timeit.timeit(lambda: interpolator.regrid(xfield[myvar[0]]), number = nr)
    #arrnomask = timeit.timeit(lambda: interpolator.regrid(xfield[myvar[0]], masked = False).load(), number = nr)
    
    setwrite =  timeit.timeit(lambda: interpolator.regrid(xfield).to_netcdf('test.nc'), number = nr)
    if os.path.isfile('test.nc'):
        os.remove('test.nc')
    arrwrite = timeit.timeit(lambda: interpolator.regrid(xfield[myvar[0]]).to_netcdf('test2.nc'), number = nr)
    if os.path.isfile('test2.nc'):
        os.remove('test2.nc')
    data.append([nvars, nrecords, ccdo, cdonoload, sset, arr, arrnoload, setwrite, arrwrite])


cnames = ['NVars', 'NRecords', 'CDO', 'CDO (NoLoad)',
          'SMM (Dataset)', 'SMM (DataArray)', 'SMM (DataArray+NoLoad)', 
          'SMM (Dataset+Write)', 'SMM (DataArray+Write)']
df = pd.DataFrame(data, index = base.keys(), columns = cnames)
final = pd.concat([df.iloc[:,0:2],df.iloc[:,2:].div(df[cnames[2]], axis=0)], join='outer', axis=1)
final



Analyzing nemo-eORCA12_hpz10_nested_oce.nc with options {'methods': ['nn', 'con', 'bil'], 'accesses': ['Dataset', 'DataArray'], 'extra': '', 'chunks': None, 'options': '--force', 'var': ['avg_sos']}
Using variable ['avg_sos'] for regridding


,NVars,NRecords,CDO,CDO (NoLoad),SMM (Dataset),SMM (DataArray),SMM (DataArray+NoLoad),SMM (Dataset+Write),SMM (DataArray+Write)
nemo-eORCA12_hpz10_nested_oce.nc,1,"(12582912,)",1.0,1.015674,1.632431,1.628491,0.165671,1.670202,1.659984


In [34]:
df

,NVars,NRecords,CDO,CDO (NoLoad),SMM (Dataset),SMM (DataArray),SMM (DataArray+NoLoad),SMM (Dataset+Write),SMM (DataArray+Write)
nemo-eORCA12_hpz10_nested_oce.nc,1,"(12582912,)",0.481526,0.461839,0.846337,0.782284,0.078697,0.792834,0.780966
